In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project folder containing datasets.")

OULAD_DIR = PROJECT_ROOT / "datasets" / "OULAD(Primary dataset)"
students = pd.read_csv(OULAD_DIR / "studentInfo.csv", na_values=["?"])

print("Student enrollment rows:", len(students))
display(students.head())

In [ ]:
print("Missing values:")
display(students.isna().sum().sort_values(ascending=False))

print("Final result counts:")
display(students["final_result"].value_counts(dropna=False))

students["final_result"].value_counts().plot(kind="bar", title="Student final results")
plt.xlabel("Final result")
plt.ylabel("Number of enrollments")
plt.tight_layout()
plt.show()

In [ ]:
for column in ["gender", "age_band", "highest_education"]:
    print(f"Final results by {column} (row percentages):")
    display(
        (pd.crosstab(students[column], students["final_result"], normalize="index") * 100)
        .round(1)
    )

In [ ]:
keys = ["code_module", "code_presentation", "id_student"]
vle_path = OULAD_DIR / "studentVle.csv"
click_parts = []

for chunk in pd.read_csv(
    vle_path,
    usecols=keys + ["sum_click"],
    chunksize=500_000,
):
    click_parts.append(
        chunk.groupby(keys, as_index=False)["sum_click"].sum()
    )

clicks = (
    pd.concat(click_parts, ignore_index=True)
    .groupby(keys, as_index=False)["sum_click"]
    .sum()
)

analysis = students.merge(
    clicks,
    on=keys,
    how="left",
    validate="one_to_one",
)
analysis["sum_click"] = analysis["sum_click"].fillna(0)

print("Total VLE clicks per enrollment:")
display(analysis["sum_click"].describe())

In [ ]:
analysis["click_group"] = pd.qcut(
    analysis["sum_click"],
    q=4,
    duplicates="drop",
)

result_by_click_group = (
    pd.crosstab(
        analysis["click_group"],
        analysis["final_result"],
        normalize="index",
    ) * 100
).round(1)

display(result_by_click_group)

In [ ]:
display(
    analysis.groupby("click_group", observed=True).agg(
        enrollments=("id_student", "size"),
        median_clicks=("sum_click", "median"),
    )
)

zero_clicks = analysis["sum_click"].eq(0)
print(f"Zero-click enrollments: {zero_clicks.sum()} ({zero_clicks.mean():.1%})")

ax = result_by_click_group.plot(kind="bar", stacked=True, figsize=(9, 5))
ax.set_title("Final result by VLE click group")
ax.set_xlabel("VLE clicks per enrollment")
ax.set_ylabel("Percentage")
ax.legend(title="Final result", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
by_module = (
    pd.crosstab(
        [analysis["code_module"], analysis["click_group"]],
        analysis["final_result"],
        normalize="index",
    ) * 100
).round(1)

display(by_module)

In [ ]:
module_group_counts = (
    analysis.groupby(["code_module", "click_group"], observed=True)
    .size()
    .unstack(fill_value=0)
)

display(module_group_counts)